# Custom CNN architecture

In [ ]:
# Run the previous notebook to load all its classes and functions
%run cbis_ddsm_create_ROI_dataset_classification_rs3.ipynb

Processing calc patients:  43%|███████████████████████▏                              | 226/527 [00:02<00:04, 74.61it/s]

In [ ]:
X_train = X_train_non_clahe.astype('float32') / 255.0
X_val   = X_val_non_clahe.astype('float32') / 255.0
X_test  = X_test_non_clahe.astype('float32') / 255.0

print(f"Final shapes AFTER NORMALIZATION -> X_train_non_clahe: {X_train.shape}, X_val_non: {X_val.shape}, X_test_non: {X_test.shape}")


## ON-THE-FLY augmentation

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_score, recall_score, f1_score

from tensorflow.keras import layers
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input, LeakyReLU, BatchNormalization, Conv2D, MaxPooling2D, Flatten,  Dropout, Activation
from tensorflow.keras.utils import plot_model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.losses import SparseCategoricalCrossentropy
import tensorflow as tf
from tensorflow.keras.datasets import mnist, cifar10
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

import seaborn as sns
import time

data_augmentation = tf.keras.Sequential(
    [
        layers.RandomRotation(
            factor=(-0.5, 0.5), 
            fill_mode="reflect"
        ),

    ],
    name="data_augmentation",
)



In [ ]:
print(f"Final shapes AFTER NORMALIZATION -> y_train_non_clahe: {y_train_non_clahe.shape}")
print(f"Final shapes AFTER NORMALIZATION -> y_val_non_clahe: {y_val_non_clahe.shape}")

In [ ]:
#Apply early stopping to save time & avoid overfitting
# Early stopping
early_stop = EarlyStopping(patience=8, restore_best_weights=True,monitor='val_loss', verbose=1)


    # Model definition
model = Sequential([
        Input(shape=(224, 224, 1)),
        data_augmentation,
        Conv2D(32, kernel_size=(3,3), activation='relu'),
        MaxPooling2D(pool_size=(2, 2)),

        Conv2D(32*2, kernel_size=(3,3), activation='relu'),
        MaxPooling2D(pool_size=(2, 2)),

        Conv2D(32*4, kernel_size=(3,3), activation='relu'),
        MaxPooling2D(pool_size=(2, 2)),

        Flatten(),
        Dense(128, activation='relu'),
        Dropout(0.5),
        Dense(1, activation='sigmoid')
    ])

    # Compile
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

    # Train
model.fit(
        X_train, y_train_non_clahe,
        validation_data=(X_val, y_val_non_clahe),
        epochs=100,
        batch_size=8,
        callbacks=[early_stop]
    )



In [ ]:
from sklearn.metrics import fbeta_score


# Predict probabilities
y_val_probs = model.predict(X_val)

# Convert probabilities to class labels
y_val_pred = (y_val_probs >= 0.5).astype(int).flatten()

# Classification Report (includes precision, recall, f1-score per class)
print("\nClassification Report:")
print(model.name)
print(classification_report(y_val_non_clahe, y_val_pred, digits=4))


f2_score = fbeta_score(y_val_non_clahe, y_val_pred, beta=2, average='weighted')
print("F2-score:", round(f2_score, 4))


cm = confusion_matrix(y_val_non_clahe, y_val_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Calc', 'Mass'], yticklabels=['Calc', 'Mass'])
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix (Val Set)')
plt.show()



# Explaining important conceps

### CNNs
- specialized NNs designed for grid-like data
- automatically learn spacial hierarchies of features through convolution operations
- highly effective in tasks involving visual perception

### Convolutional layer
**Convolution** is a mathematical operation where a small filter (kernel) is systematically applied across an input (such as an image) to produce a feature map that captures important local patterns like edges, textures, or more complex structures.

- **number of filters** = how many different kernels does the layer apply? So we basically get **number of filters** different feature maps.
- As we go on through the layers of the model, we may need more and more different kernels to try, because we have more intricate patterns in the images.

- **kernel_size** = what is the size of the filter/kernels, helping us to detect edges like shapes, edges etc.

- **activation** = the activation function used

### Pooling layers

A pooling layer reduces the spatial dimentions(width and height) of a feature map by summarizing regions of th input, helping to decrease computation, control overfitting, and make the network more robust to small translations

Purpose:

- downsampling: reduces the size of feature maps;
- feature preservation: keeps the most important information
- translation invariance: small changes in input data do not change the pooled output much

### Max Pooling

Helps us detect the strongest activation, e.g. what feature is the most present. This way, we can detect more proeminent patterns and also improve computations.

### Fully Connected Layers

Learn from the high-level features extracted by convolutional and pooling layers

- **dropout** - during training, turns off a percent of neurons in the Dense layer, to prevent overfitting.

### ReLU Activation
- keeps patterns of the data, also getting rid of negative values
- neurons stuck with negative inputs stop updating (gradient = 0)
- most popular in modern NNs

### LeakyReLU 
- allows a small gradient when x < 0 → avoids dying ReLU.

### Batch size
- how many training samples the network processes before updating its weights during training. It influences both the model's performance and the computational efficiency.
- some mostly used values are 32, 64, 128 ...
- I chose 32 because it puts in balance the computational resources needed and the efficiency of the model



### Adam Optimizer

Adam (short for **Adaptive Moment Estimation**) is one of the most popular and effective optimization algorithms used to train deep learning models. Adam adapts the learning rate for each parameter individually using **first** and **second moments** of the gradients:

- The **first moment** is the **mean** of the gradient (like momentum).
- The **second moment** is the **uncentered variance** of the gradient.

This helps Adam:
- Converge faster
- Handle sparse gradients
  